# Banking & Account Management System

In [1]:
from abc import ABC,abstractmethod
class InvalidAmountError(Exception):
    def __init__(self,amount):
        super().__init__(f'{amount} is invalid it should be a positive value')
class InsufficientBalanceError(Exception):
    pass
class SameAccountTransferError(Exception):
    pass
class Person:
    def __init__(self,name,age,phone_number):
        self.name=name
        self.age=age
        self.phone_number=phone_number
class Customer(Person):
    def __init__(self,customer_id,address,name,age,phone_number):
        self.customer_id=customer_id
        self.address=address
        super().__init__(name,age,phone_number)
class Account(ABC):
    def __init__(self,account_number,customer,balance):
        self.account_number=account_number
        self.customer=customer
        self._balance=balance
        self.transactions=[]
        self.transaction_count=0
    @property
    def balance(self):
        return self._balance
    @balance.setter
    def balance(self,amount):
        if not isinstance(amount,(int,float)):
            raise ValueError('Invalid amount type')
        self._balance=amount
    @abstractmethod
    def deposit(self,amount):
        pass
    @abstractmethod
    def withdraw(self,amount):
        pass
    def add_transaction(self,transaction_type,amount):
        self.transaction_count+=1
        transaction=Transaction(self.transaction_count,transaction_type,amount,self.balance)
        self.transactions.append(transaction)
    def display_transactions(self):
        for transaction in self.transactions:
            print(f'Transaction ID: {transaction.transaction_id}')
            print(f'Transaction Type: {transaction.transaction_type}')
            print(f'Amount: {transaction.amount}')
            print(f'Balance: {transaction.balance}')
    def __str__(self):
        return f'Account Number: {self.account_number}\nCustomer: {self.customer.name}\nBalance: {self.balance}'
    def __eq__(self,other):
        return self.account_number==other.account_number
    def __repr__(self):
        return f'{self.__class__.__name__}(account_number={self.account_number}, balance={self.balance})'
class SavingsAccount(Account):
    def __init__(self,account_number,customer,balance,interest_rate):
        super().__init__(account_number,customer,balance)
        self.interest_rate=interest_rate
    def deposit(self,amount):
        if not isinstance(amount,int):
            raise InvalidAmountError(amount)
        if amount<=0:
            raise InvalidAmountError(amount)
        self.balance+=amount
        self.add_transaction('deposit',amount)
    def withdraw(self,amount):
        if not isinstance(amount,int):
            raise InvalidAmountError(amount)
        if amount<=0:
            raise InvalidAmountError(amount)
        if amount>self.balance:
            raise InsufficientBalanceError('Amount cannot be greater than balance')
        self.balance-=amount
        self.add_transaction('withdrawal',amount)
    def calculate_interest(self):
        return self.balance*(self.interest_rate/100)
class CurrentAccount(Account):
    def __init__(self,account_number,customer,balance,overdraft_limit):
        super().__init__(account_number,customer,balance)
        self.overdraft_limit=overdraft_limit
    def deposit(self,amount):
        if not isinstance(amount,int):
            raise InvalidAmountError(amount)
        if amount<=0:
            raise InvalidAmountError(amount)
        self.balance+=amount
        self.add_transaction('deposit',amount)
    def withdraw(self,amount):
        if not isinstance(amount,int):
            raise InvalidAmountError(amount)
        if amount<=0:
            raise InvalidAmountError(amount)
        if amount>self.balance+self.overdraft_limit:
            raise InsufficientBalanceError('Amount cannot be greater than balance')
        self.balance-=amount
        self.add_transaction('withdrawal',amount)
class Transaction:
    def __init__(self,transaction_id,transaction_type,amount,balance):
        self.transaction_id=transaction_id
        self.transaction_type=transaction_type
        self.amount=amount
        self.balance=balance
class Bank:
    def __init__(self,name):
        self.name=name
        self.customers=[]
        self.accounts=[]
    def add_customer(self,customer):
        if customer in self.customers:
            raise ValueError('Customer already registered')
        self.customers.append(customer)
    def add_account(self,account):
        if account in self.accounts:
            raise ValueError('Account already exists')
        self.accounts.append(account)
    def find_customer(self,customer_id):
        for customer in self.customers:
            if customer.customer_id==customer_id:
                return customer
        raise ValueError('Customer not found')
    def find_account(self,account_number):
        for account in self.accounts:
            if account.account_number==account_number:
                return account
        raise ValueError('Account not found')
    def transfer(self, from_account_number, to_account_number, amount):
        if from_account_number==to_account_number:
            raise SameAccountTransferError('cant transfer to same account')
        sender=self.find_account(from_account_number)
        receiver=self.find_account(to_account_number)
        sender.withdraw(amount)
        receiver.deposit(amount)
        print('Successfully transfered money')
# ==============================
# TESTING BANKING SYSTEM
# ==============================

# Create Bank
bank = Bank("ABC Bank")


# ==============================
# CREATE CUSTOMERS
# ==============================

customer1 = Customer(
    101,
    "Hyderabad",
    "Rahul",
    25,
    "9876543210"
)

customer2 = Customer(
    102,
    "Vijayawada",
    "Priya",
    28,
    "9123456780"
)


# ==============================
# ADD CUSTOMERS
# ==============================

bank.add_customer(customer1)
bank.add_customer(customer2)

print("Customers registered successfully")


# ==============================
# CREATE ACCOUNTS
# ==============================

savings = SavingsAccount(
    1001,
    customer1,
    10000,
    6
)

current = CurrentAccount(
    1002,
    customer2,
    5000,
    2000
)


# ==============================
# ADD ACCOUNTS
# ==============================

bank.add_account(savings)
bank.add_account(current)

print("Accounts created successfully")


# ==============================
# DISPLAY ACCOUNTS
# ==============================

print("\n--- SAVINGS ACCOUNT ---")
print(savings)

print("\n--- CURRENT ACCOUNT ---")
print(current)


# ==============================
# DEPOSIT
# ==============================

print("\n--- DEPOSIT ---")

savings.deposit(2000)
current.deposit(1000)

print("Savings balance:", savings.balance)
print("Current balance:", current.balance)


# ==============================
# WITHDRAW
# ==============================

print("\n--- WITHDRAW ---")

savings.withdraw(3000)
current.withdraw(2000)

print("Savings balance:", savings.balance)
print("Current balance:", current.balance)


# ==============================
# CURRENT ACCOUNT OVERDRAFT
# ==============================

print("\n--- OVERDRAFT TEST ---")

current.withdraw(5000)

print("Current balance after overdraft:", current.balance)


# ==============================
# INTEREST CALCULATION
# ==============================

print("\n--- INTEREST ---")

interest = savings.calculate_interest()

print("Savings balance:", savings.balance)
print("Interest:", interest)


# ==============================
# MONEY TRANSFER
# ==============================

print("\n--- MONEY TRANSFER ---")

bank.transfer(
    1001,
    1002,
    1000
)

print("Savings balance:", savings.balance)
print("Current balance:", current.balance)


# ==============================
# TRANSACTION HISTORY
# ==============================

print("\n--- SAVINGS TRANSACTIONS ---")

savings.display_transactions()


print("\n--- CURRENT TRANSACTIONS ---")

current.display_transactions()


# ==============================
# FIND CUSTOMER
# ==============================

print("\n--- FIND CUSTOMER ---")

customer = bank.find_customer(101)

print(customer.name)
print(customer.phone_number)


# ==============================
# FIND ACCOUNT
# ==============================

print("\n--- FIND ACCOUNT ---")

account = bank.find_account(1001)

print(account)


# ==============================
# __repr__ TEST
# ==============================

print("\n--- REPR TEST ---")

print(repr(savings))
print(repr(current))


# ==============================
# __eq__ TEST
# ==============================

print("\n--- EQUALITY TEST ---")

another_savings = SavingsAccount(
    1001,
    customer1,
    5000,
    6
)

print(savings == another_savings)


# ==============================
# INVALID AMOUNT TEST
# ==============================

print("\n--- INVALID AMOUNT TEST ---")

try:
    savings.deposit(-500)

except InvalidAmountError as e:
    print("Error:", e)


# ==============================
# INSUFFICIENT BALANCE TEST
# ==============================

print("\n--- INSUFFICIENT BALANCE TEST ---")

try:
    savings.withdraw(100000)

except InsufficientBalanceError as e:
    print("Error:", e)


# ==============================
# SAME ACCOUNT TRANSFER TEST
# ==============================

print("\n--- SAME ACCOUNT TRANSFER TEST ---")

try:
    bank.transfer(1001, 1001, 500)

except SameAccountTransferError as e:
    print("Error:", e)

    
        


Customers registered successfully
Accounts created successfully

--- SAVINGS ACCOUNT ---
Account Number: 1001
Customer: Rahul
Balance: 10000

--- CURRENT ACCOUNT ---
Account Number: 1002
Customer: Priya
Balance: 5000

--- DEPOSIT ---
Savings balance: 12000
Current balance: 6000

--- WITHDRAW ---
Savings balance: 9000
Current balance: 4000

--- OVERDRAFT TEST ---
Current balance after overdraft: -1000

--- INTEREST ---
Savings balance: 9000
Interest: 540.0

--- MONEY TRANSFER ---
Successfully transfered money
Savings balance: 8000
Current balance: 0

--- SAVINGS TRANSACTIONS ---
Transaction ID: 1
Transaction Type: deposit
Amount: 2000
Balance: 12000
Transaction ID: 2
Transaction Type: withdrawal
Amount: 3000
Balance: 9000
Transaction ID: 3
Transaction Type: withdrawal
Amount: 1000
Balance: 8000

--- CURRENT TRANSACTIONS ---
Transaction ID: 1
Transaction Type: deposit
Amount: 1000
Balance: 6000
Transaction ID: 2
Transaction Type: withdrawal
Amount: 2000
Balance: 4000
Transaction ID: 3
Tr